In [194]:
import pandas as pd
import numpy as np
import xmlrpc.client
import io
import base64
import os
from dotenv import load_dotenv

## 1. Credenciales de ingreso

In [195]:
# Para claves de entorno instalar la libreria pip install python-dotenv
load_dotenv() # Cargar las variables de entorno desde el archivo .env
url =  os.getenv("ODOO_URL")
username =  os.getenv("ODOO_USERNAME")
password = os.getenv("ODOO_PASSWORD")
db = os.getenv("ODOO_DB")

## 2. Objecto de identificador y manipulacion

In [196]:
# Crear objecto - Para conectar
common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")

# Identificarme
uid = common.authenticate(db, username, password, {})
print(f"uid conectado : {uid}")

# Crear objecto  - Para poder hacer consultas al modelo

models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)

uid conectado : 20


## 3. Dim_Producto

In [197]:
def limpando_producto(modelo_extraido):
    # Para limpiar los diccionarios con ID + Valor solo quedarnos con el valor
    df = []

    for diccionario in modelo_extraido:
        nuevo_diccionario = {}

        for clave, valor in diccionario.items():
            if isinstance(valor, list) and len(valor) == 2:
                nuevo_diccionario[clave] = valor[1]

            elif valor is False:
                nuevo_diccionario[clave] = None

            else:
                nuevo_diccionario[clave] = valor

        df.append(nuevo_diccionario)

    # Limpiando el DF 
    df = pd.DataFrame(df)
    df[["Familia", "Familia_2", "Familia_3"]] = df["categ_id"].str.split("/", n=2, expand=True)
    df = df.rename(
        columns={
            "name" : "Producto", "numero_parte" : "Nro_Parte", "product_brand_id" : "Marca", 
            "default_code" : "Cod_Producto", "uom_id": "Unidad_Medida", "tipo_uso": "Uso"
            }
        )

    df =  df.drop(columns=["categ_id"])
    df = df[["Cod_Producto", "Producto", "Nro_Parte", "Marca", "Unidad_Medida", "Familia", "Familia_2", "Familia_3", "Uso"]]

    cod_proyecto = ["G2020382","G2020383","G2011309","G2011310","G2290040","G2290039","G2290041","G2290038","G2290037","G2290036",
                    "G2011308","G2011311","G2230040","G2020385","G2210081","G2020384","G2210079","G2210080","G2210078","G2230041","G2230042"]
    condiciones = [
        df["Cod_Producto"].isin(cod_proyecto),
        df["Producto"].str.contains("TA0", case=False, na=False)
    ]

    resultado = ["Proyecto Chasis", "Proyecto Chasis"]

    df["Tipo_Proyecto"] = np.select(
        condiciones, resultado, default="General"
    )
    # Quitar espacios en blanco al inicio y al final de las las columnas de Familias
    df['Familia'] = df['Familia'].str.strip()
    df['Familia_2'] = df['Familia_2'].str.strip()
    df['Familia_3'] = df['Familia_3'].str.strip()

    return df


In [198]:
"""
    1. CONEXION A DF 
"""
productos = models.execute_kw(
    db, uid, password,
    "product.template",
    "search_read",
    [
        [
            ["default_code", "!=", False]
        ]
    ],
    {
        "fields" : ["name", "numero_parte", "product_brand_id", "default_code",
                    "categ_id", "uom_id", "tipo_uso"],
        "context" :{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
            "search_default_consumable": 1
        }
    }
)

"""
    2. LIMPIEZA DF
"""
df_productos = limpando_producto(productos)

df_productos.to_excel("Dim_Producto.xlsx", index=False)

## 4. Orden de Trabajo

### Modulo Anterior OT

In [199]:
report_ots_tatsa =  models.execute_kw(
    db, uid, password,
    "orden.trabajo",
    "search_read",
    [
        [
            ["state", "!=", "cancel"],
            ["fecha_creacion", ">", "2024-12-31"]
        ]
    ],
    {
        "fields": ["name", "tipo_mantenimiento", "flota", "nro_ot_formatoimpreso", "nombre_flota"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
        },
        #"limit": 20,
    }
)

# Limpiar las lista en columnas 
df = []

for diccionario in report_ots_tatsa:
    nuevo_diccionario = {}
    
    for clave, valor in diccionario.items():
        if isinstance(valor, list) and len(valor) == 2:
            nuevo_diccionario[clave] = valor[1]
        
        elif valor is False:
            nuevo_diccionario[clave] = None
        
        else:
            nuevo_diccionario[clave] = valor

    df.append(nuevo_diccionario)

df_report_ots_tatsa = pd.DataFrame(df)
df_report_ots_tatsa = df_report_ots_tatsa[df_report_ots_tatsa["name"].notna()]
df_report_ots_tatsa["tipo_mantenimiento"] = df_report_ots_tatsa["tipo_mantenimiento"].replace(
    {
        "correctivo" : "Correctivo",
        "correctivo_reactivo" : "Correctivo",
        "correctivo_programado" : "Correctivo",
        "preventivo_programado" : "Preventivo",
        "preventivo" : "Preventivo",
        "predictivo" : "Preventivo", 
        "preventivo_linea": "Preventivo",
        "predictivo_rutina" : "Preventivo",
        False : np.nan
    }
)
df_report_ots_tatsa["flota"] = df_report_ots_tatsa["flota"].str.extract(r'\((.*?)\)')

condiciones_tamaño = [
    df_report_ots_tatsa["nombre_flota"] == "Mercedes/Class A",
    df_report_ots_tatsa["nombre_flota"] == "Opel/Zafira",
    df_report_ots_tatsa["nombre_flota"] == "Peugeot/BIPPER",
    df_report_ots_tatsa["nombre_flota"] == "Toyota/Hilux -",
    df_report_ots_tatsa["nombre_flota"] == "KING LONG/XMQ6892G 2R",
    df_report_ots_tatsa["nombre_flota"] == "KING LONG/XMQ6892G 1R",
    df_report_ots_tatsa["nombre_flota"] == "KING LONG/XMQ6892G",
    df_report_ots_tatsa["nombre_flota"] == "KING LONG/-XMQ6181G",
    df_report_ots_tatsa["nombre_flota"] == "KING LONG/XMQ6181G",
    df_report_ots_tatsa["nombre_flota"] == "KING LONG/XMQ6123G 1R",
    df_report_ots_tatsa["nombre_flota"] == "KING LONG/XMQ6123G",
    df_report_ots_tatsa["nombre_flota"] == "KING LONG/SLK 6905"
]

resultados_tamaño = [
    "CHASIS/MERCEDES", 
    "CONVERSION",
    "CAMIONETA",
    "CAMIONETA",
    "9 METROS",
    "9 METROS",
    "9 METROS",
    "AL-080",
    "18 METROS",
    "12 METROS",
    "12 METROS",
    "9 METROS"
]

df_report_ots_tatsa["nombre_flota"] = np.select(
    condiciones_tamaño, resultados_tamaño, default=df_report_ots_tatsa["nombre_flota"])

df_report_ots_tatsa["flota"] = np.where(
    df_report_ots_tatsa["flota"]=="CONVERSION", 
    df_report_ots_tatsa["nro_ot_formatoimpreso"], 
    df_report_ots_tatsa["flota"]
)

df_report_ots_tatsa["nro_ot_formatoimpreso"] = df_report_ots_tatsa["nro_ot_formatoimpreso"].str.replace(
    r"^([A-Z0-9]{17})\s*/.*$", r"\1", regex=True)

condicion = (
    (df_report_ots_tatsa["flota"] == "CHASIS/MERCEDES") &
    (df_report_ots_tatsa["nro_ot_formatoimpreso"].notna()) &
    (df_report_ots_tatsa["nro_ot_formatoimpreso"].str.strip() != "")
)

df_report_ots_tatsa["flota"] = np.where(
    condicion,
    df_report_ots_tatsa["nro_ot_formatoimpreso"],
    df_report_ots_tatsa["flota"]
)

df_report_ots_tatsa = df_report_ots_tatsa.rename(
    columns={
        "name": "Cod_Orden_Trabajo", 
        "tipo_mantenimiento": "Tipo_Mantto", 
        "flota": "Cod_Bus_Orden_Trabajo",
        "nombre_flota": "Tamaño_Bus"
    }
)

df_report_ots_tatsa = df_report_ots_tatsa[["Cod_Orden_Trabajo", "Tipo_Mantto", "Cod_Bus_Orden_Trabajo", "Tamaño_Bus"]]
df_report_ots_tatsa = df_report_ots_tatsa.drop_duplicates()

### Modulo de Averias

In [200]:
report_averias_tatsa =  models.execute_kw(
    db, uid, password,
    "maint.breakdown",
    "search_read",
    [
        [
            ["reason_id", "ilike", "LIMABUS"]
        ]
    ],
    {
        "fields": ["request_id", "swap_serie", "reason_id"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
        },
        #"limit": 5,
    }
)

# Limpiar las lista en columnas 
df = []

for diccionario in report_averias_tatsa:
    nuevo_diccionario = {}
    
    for clave, valor in diccionario.items():
        if isinstance(valor, list) and len(valor) == 2:
            nuevo_diccionario[clave] = valor[1]
        
        elif valor is False:
            nuevo_diccionario[clave] = None
        
        else:
            nuevo_diccionario[clave] = valor

    df.append(nuevo_diccionario)

df_report_averias_tatsa = pd.DataFrame(df)
df_report_averias_tatsa["request_id"] = df_report_averias_tatsa["request_id"].str.split(" - ").str[0]
df_report_averias_tatsa = df_report_averias_tatsa[df_report_averias_tatsa["swap_serie"].notna()]
df_report_averias_tatsa = df_report_averias_tatsa[df_report_averias_tatsa["swap_serie"] != "NO TIENE PLACA"]
df_report_averias_tatsa = df_report_averias_tatsa.rename(
    columns={"request_id" : "Cod_Orden_Trabajo", "swap_serie" : "Cod_Bus_Averia", "reason_id" : "Tamaño_Bus"}
)
df_report_averias_tatsa = df_report_averias_tatsa[["Cod_Orden_Trabajo", "Cod_Bus_Averia", "Tamaño_Bus"]]
df_report_averias_tatsa = df_report_averias_tatsa.drop_duplicates(
    subset=["Cod_Orden_Trabajo"], keep="first"
)

### Modulo NEW OT

In [201]:
report_ots_New_tatsa =  models.execute_kw(
    db, uid, password,
    "maintenance.request",
    "search_read",
    [
        [
            #["create_date", ">", "2024-12-31"]
        ]
    ],
    {
        "fields": ["name", "maintenance_type"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
        },
        #"limit": 20,
    }
)

df_report_ots_New_tatsa = pd.DataFrame(report_ots_New_tatsa)
df_report_ots_New_tatsa = df_report_ots_New_tatsa[["name", "maintenance_type"]]
df_report_ots_New_tatsa = df_report_ots_New_tatsa.rename(columns={"name": "Cod_Orden_Trabajo", "maintenance_type" : "Tipo_Mantto_New"})
df_report_ots_New_tatsa["Tipo_Mantto_New"] = df_report_ots_New_tatsa["Tipo_Mantto_New"].replace(
    {
        "corrective" : "Correctivo",
        "preventive" : "Preventivo",
        "predictive" : "Preventivo"
    }
)

## 5. Transferencias

### Transferencias

In [202]:
#### Funcion de limpieza de transferencia
def limpando_transferencias(modelo_extraido):
    # Para limpiar los diccionarios con ID + Valor solo quedarnos con el valor
    df = []

    for diccionario in modelo_extraido:
        nuevo_diccionario = {}

        for clave, valor in diccionario.items():
            if isinstance(valor, list) and len(valor) == 2:
                nuevo_diccionario[clave] = valor[1]

            elif valor is False:
                nuevo_diccionario[clave] = None

            else:
                nuevo_diccionario[clave] = valor

        df.append(nuevo_diccionario)

    # Limpiando el DF 
    df = pd.DataFrame(df)
    df["flota_name"] = df["flota_name"].str.extract(r'\((.*?)\)')
    df["vehicle_ot_id"] = df["vehicle_ot_id"].str.extract(r'\((.*?)\)')
    df["ots_origenes_name"] = df["ots_origenes_name"].str.extract(r"'(.*?)'")
    df["Cod_Bus"] = df["flota_name"].fillna(df["vehicle_ot_id"])
    df["Cod_Orden_Trabajo"] = df["ots_origenes_name"].fillna(df["request_ot"])
    df = df[["name", "Cod_Bus", "Cod_Orden_Trabajo"]]
    df = df.rename(columns={"name" : "Cod_Albaran"})
    return df 

"""
    Extrayendo el modelo Odoo
"""
report_transferencias_tatsa =  models.execute_kw(
    db, uid, password,
    "stock.picking",
    "search_read",
    [
        [
            ["kardex_date", "!=", False]
        ]
    ],
    {
        "fields": ["name", "kardex_date", "flota_name", "ots_origenes_name", "request_ot", "vehicle_ot_id", "state"],
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2],
        },
        #"limit": 2,
    }
)

"""
    Aplicando funcion de limpieza
"""

df_report_transferencias_tatsa  = limpando_transferencias(report_transferencias_tatsa)

df_report_transferencias_tatsa = pd.merge(
    df_report_transferencias_tatsa,
    df_report_ots_tatsa[["Cod_Orden_Trabajo", "Tipo_Mantto", "Cod_Bus_Orden_Trabajo"]],
    on="Cod_Orden_Trabajo",
    how="left"
)

df_report_transferencias_tatsa["Cod_Bus"] = df_report_transferencias_tatsa["Cod_Bus_Orden_Trabajo"].fillna(df_report_transferencias_tatsa["Cod_Bus"])
df_report_transferencias_tatsa = df_report_transferencias_tatsa.drop(columns=["Cod_Bus_Orden_Trabajo"])

df_report_transferencias_tatsa = pd.merge(
    df_report_transferencias_tatsa,
    df_report_averias_tatsa[["Cod_Orden_Trabajo", "Cod_Bus_Averia"]],
    on="Cod_Orden_Trabajo",
    how="left"
)

df_report_transferencias_tatsa["Cod_Bus"] = df_report_transferencias_tatsa["Cod_Bus_Averia"].fillna(df_report_transferencias_tatsa["Cod_Bus"])
df_report_transferencias_tatsa = df_report_transferencias_tatsa.drop(columns=["Cod_Bus_Averia"])

df_report_transferencias_tatsa = pd.merge(
    df_report_transferencias_tatsa,
    df_report_ots_New_tatsa[["Cod_Orden_Trabajo", "Tipo_Mantto_New"]],
    on="Cod_Orden_Trabajo",
    how="left"
)

df_report_transferencias_tatsa["Tipo_Mantto"] = df_report_transferencias_tatsa["Tipo_Mantto_New"].fillna(df_report_transferencias_tatsa["Tipo_Mantto"])
df_report_transferencias_tatsa = df_report_transferencias_tatsa.drop(columns=["Tipo_Mantto_New"])

correcciones_bus = {
    "ASUNSTTA01426" : "COA-145",
    "ASUNSTTA01463" : "COA-198",
    "ASUNSTTA01465" : "COA-189",
    "ASUNSTTA01467" : "COA-187-FAKE",
    "ASUNSTTA01493" : "COA-188",
    "ASUNSTTA01495" : "COA-183",
    "ASUNSTTA01502" : "COA-067",
    "ASUNSTTA01511" : "COA-043",
    "ASUNSTTA01515" : "COA-101",
    "ASUNSTTA01517" : "COA-086",
    "ASUNSTTA01518" : "COA-086",
    "ASUNSTTA01533" : "COA-030",
    "ASUNSTTA01550" : "COA-105",
    "ASUNSTTA01557" : "COA-105",
    "ASUNSTTA01558" : "COA-186",
    "ASUNSTTA01577" : "COA-194",
    "ASUNSTTA01578" : "COA-176",
    "ASUNSTTA01579" : "COA-114",
    "ASUNSTTA01623" : "COA-151",
    "ASUNSTTA01639" : "COA-170"
}
df_report_transferencias_tatsa["Cod_Bus"] = df_report_transferencias_tatsa["Cod_Albaran"].map(correcciones_bus).fillna(df_report_transferencias_tatsa["Cod_Bus"])


## 6. Reporte Inventario

### Funcion de Serie de Transferencias

In [203]:
"""
    1. PARA EXTRAER EL ID DEL PERIODO DE WIZARD REPORTE DE INVENTARIO
"""

periodos = models.execute_kw(
    db, uid, password,
    "account.period.kardex",
    "search_read",
    [[]],
    {   
        "fields":["id", "code", "display_name"],
        "context": {
            "lang": "es_PE", 
            "tz": "America/Lima",
            "allowed_company_ids": [1]
        }
    }
)

df_periodo_reporte_inv = pd.DataFrame(periodos) #El id es necesario 
df_periodo_reporte_inv[df_periodo_reporte_inv["code"]=="202612"]

,id,code,display_name
55,62,202612,202612


### Tatsa 2025

In [204]:
"""
    2. GENERAR EL WIZARD CON EL FILTRO PERIDO
"""

wizard_id_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "create",
    [
        {
            "fecha_inicio": 34, # Periodo 202501
            "fecha_final" : 45, # Periodo 202512
            "product_ids": [(6, 0, [])], # Sin filtro Repuestos
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)

print("Wizard creado:", wizard_id_reporte)

"""
    3. EJECUTAR EL WIZARD CREADO
"""
accion_wizard_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "descargar_report",
    [
        [wizard_id_reporte]
    ],
    {
        "context":{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)


popup_id = accion_wizard_reporte["res_id"]
print(f"Id creado del wizard {popup_id}")

"""
    4. LEER Y EJECUTAR EL WIZARD Y EXTRAER EL DATAFRAME
"""
popup = models.execute_kw(
    db, uid, password,
    "popup.it",
    "read",
    [[popup_id]],
    {
        "fields" : ["output_name", "output_file", "display_name"],
        "context" : {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }

)[0]

print("\nInformacion Extra")

print(popup["output_name"])                  # nombre del archivo: REPORTE DE INVENTARIO.xlsx
print(type(popup["output_file"]))            # str: el Excel llega escrito como texto
print(len(popup["output_file"]))             # tamaño del texto

# 2) Texto base64 -> bytes del archivo (Se manda en texto para no descargar el archivo)
#    Odoo no puede enviar un archivo binario por XML-RPC, así que lo envía como
#    texto (base64). b64decode lo devuelve a su forma original: los bytes del .xlsx.

contenido = base64.b64decode(popup["output_file"])
print(contenido[:2]) # b'PK' -> así empieza todo .xlsx (es un zip)

# 3) Bytes -> "archivo en memoria"
#    pandas espera un archivo. BytesIO hace que esos bytes se comporten como uno,
#    sin crearlo en el disco.
archivo_en_memoria = io.BytesIO(contenido)

# 4) Excel -> tabla
#    dtype=str lee todo como texto, así un código '00123' no se convierte en 123.
df_inv = pd.read_excel(archivo_en_memoria, dtype=str, header=9)

df_reporte_inv_tat_2025 = df_inv.copy()

Wizard creado: 2054
Id creado del wizard 91526

Informacion Extra
REPORTE DE INVENTARIO.xlsx
<class 'str'>
1389556
b'PK'


### Tatsa 2026

In [205]:
"""
    2. GENERAR EL WIZARD CON EL FILTRO PERIDO
"""

wizard_id_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "create",
    [
        {
            "fecha_inicio": 51, # Periodo 202501
            "fecha_final" : 62, # Periodo 202512
            "product_ids": [(6, 0, [])], # Sin filtro Repuestos
        }
    ],
    {
        "context": {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)

print("Wizard creado:", wizard_id_reporte)

"""
    3. EJECUTAR EL WIZARD CREADO
"""
accion_wizard_reporte = models.execute_kw(
    db, uid, password,
    "report.futuro.use",
    "descargar_report",
    [
        [wizard_id_reporte]
    ],
    {
        "context":{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)


popup_id = accion_wizard_reporte["res_id"]
print(f"Id creado del wizard {popup_id}")

"""
    4. LEER Y EJECUTAR EL WIZARD Y EXTRAER EL DATAFRAME
"""
popup = models.execute_kw(
    db, uid, password,
    "popup.it",
    "read",
    [[popup_id]],
    {
        "fields" : ["output_name", "output_file", "display_name"],
        "context" : {
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }

)[0]

print("\nInformacion Extra")

print(popup["output_name"])                  # nombre del archivo: REPORTE DE INVENTARIO.xlsx
print(type(popup["output_file"]))            # str: el Excel llega escrito como texto
print(len(popup["output_file"]))             # tamaño del texto

# 2) Texto base64 -> bytes del archivo (Se manda en texto para no descargar el archivo)
#    Odoo no puede enviar un archivo binario por XML-RPC, así que lo envía como
#    texto (base64). b64decode lo devuelve a su forma original: los bytes del .xlsx.

contenido = base64.b64decode(popup["output_file"])
print(contenido[:2]) # b'PK' -> así empieza todo .xlsx (es un zip)

# 3) Bytes -> "archivo en memoria"
#    pandas espera un archivo. BytesIO hace que esos bytes se comporten como uno,
#    sin crearlo en el disco.
archivo_en_memoria = io.BytesIO(contenido)

# 4) Excel -> tabla
#    dtype=str lee todo como texto, así un código '00123' no se convierte en 123.
df_inv = pd.read_excel(archivo_en_memoria, dtype=str, header=9)

df_reporte_inv_tat_2026 = df_inv.copy()

Wizard creado: 2055
Id creado del wizard 91527

Informacion Extra
REPORTE DE INVENTARIO.xlsx
<class 'str'>
1557116
b'PK'


### Centro de Costo de la OC

In [206]:
df_oc_centro_costo = pd.concat(
    [
        df_reporte_inv_tat_2026[["ORDEN DE COMPRA", "CENTRO DE COSTO", "SKU"]],
        df_reporte_inv_tat_2025[["ORDEN DE COMPRA", "CENTRO DE COSTO", "SKU"]]
    ]
)

df_oc_centro_costo = df_oc_centro_costo[df_oc_centro_costo["ORDEN DE COMPRA"].notna()]
df_oc_centro_costo = df_oc_centro_costo.drop_duplicates()
df_oc_centro_costo = df_oc_centro_costo.rename(
    columns={"ORDEN DE COMPRA" : "Cod_Orden_Compra", "CENTRO DE COSTO" : "Cuenta Analítica", "SKU" : "Cod_Producto"}
)

## 7. Kardex

### Kardex Tatsa 2025

In [207]:
"""
    1. DATOS DEL KARDEX
"""

FECHA_INICIO = "2025-01-01"      # AAAA-MM-DD
FECHA_FIN = "2025-12-31"

contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [2],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
ubicaciones = [186, 48, 47, 115, 49, 45, 46, 22, 198, 204, 210, 179, 18, 19, 20]

"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)

"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)

"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [2]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))

"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_tatsa_2025 = pd.read_excel(archivo_en_memoria, dtype=str, header=8)

Wizard creado: 8072
Se proceso de 2025-01-01 al 2025-12-31. | Lineas procesadas: 66330 | Tiempo: 0 minutos 16 segundos
Id del adjunto: 286815
Kardex Valorado.xlsx
14385692
b'PK'


### Kardex Tatsa 2026

In [208]:
"""
    1. DATOS DEL KARDEX
"""

FECHA_INICIO = "2026-01-01"      # AAAA-MM-DD
FECHA_FIN = "2026-12-31"

contexto_kardex = {
    "lang": "es_PE",
    "tz": "America/Lima",
    "allowed_company_ids": [2],
    "tipo": "valorado",
    "type_product": "product",
    "form_view_ref": "view_make_kardex_valorado",
}

# Ubicaciones (stock.location) que marca el formulario por defecto
ubicaciones = [186, 48, 47, 115, 49, 45, 46, 22, 198, 204, 210, 179, 18, 19, 20]

"""
    2. GENERAR EL WIZARD CON EL FILTRO DE FECHAS  (el "create" de tu captura)
"""

wizard_id_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "create",
    [
        {
            "fini": FECHA_INICIO,                 # fecha inicio que eliges
            "ffin": FECHA_FIN,                    # fecha fin que eliges
            "fecha_ini_mod": FECHA_INICIO,        # copia que rellena la pantalla
            "fecha_fin_mod": FECHA_FIN,           # copia que rellena la pantalla
            "check_fecha": True,                  # casilla "filtrar por fechas"
            "moneda": "pen",                      # soles
            "alllocations": True,                 # todas las ubicaciones
            "location_ids": [(6, 0, ubicaciones)],# (6,0,lista) = "usa exactamente esta lista"
            "allproducts": True,                  # todos los productos
            "products_ids": [(6, 0, [])],         # lista vacía = sin filtro de productos
            "analizador": False,
            "check_account": False,
        }
    ],
    {"context": contexto_kardex}
)

print("Wizard creado:", wizard_id_kardex)

"""
    3. EJECUTAR EL WIZARD  (el call_button "do_csvtoexcel_transvial")
"""

respuesta_kardex = models.execute_kw(
    db, uid, password,
    "make.kardex.valorado",
    "do_csvtoexcel_transvial",
    [
        [wizard_id_kardex]
    ],
    {"context": contexto_kardex}
)

# La respuesta es un diccionario. Para llegar al id del archivo vamos bajando:
#   respuesta_kardex["params"]                  -> diccionario con mensaje y botones
#   respuesta_kardex["params"]["buttons"]       -> lista de botones (la "caja")
#   respuesta_kardex["params"]["buttons"][0]    -> el primer botón (la "ficha")
#   respuesta_kardex["params"]["buttons"][0]["id"] -> id del archivo adjunto
print(str(respuesta_kardex["params"]["message"]).replace("<br/>", " | "))

adjunto_id = respuesta_kardex["params"]["buttons"][0]["id"]
print("Id del adjunto:", adjunto_id)

"""
    4. LEER EL ADJUNTO  (el Excel está guardado en ir.attachment, campo "datas")
"""

adjunto = models.execute_kw(
    db, uid, password,
    "ir.attachment",
    "read",
    [[adjunto_id]],
    {
        "fields": ["name", "datas"],
        "context": {"lang": "es_PE", "tz": "America/Lima", "allowed_company_ids": [2]},
    }
)[0]                                        # [0]: sacar la ficha de la caja

print(adjunto["name"])
print(len(adjunto["datas"]))

"""
    5. DE TEXTO A TABLA (en memoria, sin descargar)
"""

contenido_kardex = base64.b64decode(adjunto["datas"])     # texto base64 -> bytes del .xlsx
print(contenido_kardex[:2])                               # b'PK'

archivo_en_memoria = io.BytesIO(contenido_kardex)         # bytes -> "archivo" en memoria

# header=None: leemos TODO el Excel sin decidir cuál fila es el encabezado,
# para ver primero cuántas filas de títulos trae arriba.
df_kardex_tatsa_2026= pd.read_excel(archivo_en_memoria, dtype=str, header=8)

Wizard creado: 8073
Se proceso de 2026-01-01 al 2026-12-31. | Lineas procesadas: 70185 | Tiempo: 0 minutos 17 segundos
Id del adjunto: 286817
Kardex Valorado.xlsx
15130636
b'PK'


### Union Dataframe

In [209]:
def unificar_kardex(kar_tat_26, kar_tat_25, transferencia_tat, centro_costo):

    # Tatsa 2026
    df_k_tat_26 = kar_tat_26.copy()
    df_k_tat_26 = df_k_tat_26[df_k_tat_26["Sku"].notna()]
    df_k_tat_26 = df_k_tat_26[df_k_tat_26["T. OP."].notna()] #Quitamos Inventario inicial
    df_k_tat_26 = df_k_tat_26[
        ["Fecha", "Hora", "Doc. Almacen", "RUC", "Proveedor", "T. OP.", "Sku", "LOTE",
         "Costo Promedio", "Cantidad", "Importe", "Cantidad.1", "Importe.1", "Cantidad.2",
         "Importe.2", "Ubicacion Origen", "Ubicacion Destino", "Almacen", "Pedido"]]
    df_k_tat_26 = df_k_tat_26.rename(columns={"Doc. Almacen" : "Cod_Albaran"})

    # Tatsa 2025
    df_k_tat_25 = kar_tat_25.copy()
    df_k_tat_25 = df_k_tat_25[df_k_tat_25["Sku"].notna()]
    df_k_tat_25["T. OP."] = df_k_tat_25["T. OP."].fillna("16")
    df_k_tat_25 = df_k_tat_25[
        ["Fecha", "Hora", "Doc. Almacen", "RUC", "Proveedor", "T. OP.", "Sku", "LOTE",
         "Costo Promedio", "Cantidad", "Importe", "Cantidad.1", "Importe.1", "Cantidad.2",
         "Importe.2", "Ubicacion Origen", "Ubicacion Destino", "Almacen", "Pedido"]]
    df_k_tat_25 = df_k_tat_25.rename(columns={"Doc. Almacen" : "Cod_Albaran"})

    df_fin = pd.concat([df_k_tat_26, df_k_tat_25], ignore_index=True)

    df_fin = df_fin.rename(columns={"Doc. Almacen" : "Cod_Albaran"})
    df_fin = pd.merge(
        df_fin,
        transferencia_tat,
        on="Cod_Albaran",
        how="left"
    )

    df_fin["T. OP."] = pd.to_numeric(df_fin["T. OP."], errors="coerce")
    df_fin["Costo Promedio"] = pd.to_numeric(df_fin["Costo Promedio"], errors="coerce")
    df_fin["Cantidad"] = pd.to_numeric(df_fin["Cantidad"], errors="coerce")
    df_fin["Importe"] = pd.to_numeric(df_fin["Importe"], errors="coerce")
    df_fin["Cantidad.1"] = pd.to_numeric(df_fin["Cantidad.1"], errors="coerce")
    df_fin["Importe.1"] = pd.to_numeric(df_fin["Importe.1"], errors="coerce")
    df_fin["Cantidad.2"] = pd.to_numeric(df_fin["Cantidad.2"], errors="coerce")
    df_fin["Importe.2"] = pd.to_numeric(df_fin["Importe.2"], errors="coerce")
    df_fin["Fecha"] = pd.to_datetime(df_fin["Fecha"], errors="coerce")

    # unificar la columna fecha y hora
    df_fin["Fecha_Completa"] = (df_fin["Fecha"].dt.strftime('%Y-%m-%d') + ' ' + df_fin["Hora"].astype(str))

    # convierto la columna fecha completa en tipo de dato:fecha
    df_fin["Fecha_Completa"] = pd.to_datetime(df_fin["Fecha_Completa"], errors="coerce")

    df_fin = df_fin.rename(
        columns={
            "RUC" : "Ruc",
            "T. OP." : "Cod_Movimiento",
            "Sku" : "Cod_Producto",
            "LOTE" : "Lote",
            "Costo Promedio" : "Precio_Unt",
            "Cantidad" : "Cant_Ingreso",
            "Importe" : "Valor_Ingreso",
            "Cantidad.1" : "Cant_Salida",
            "Importe.1" : "Valor_Salida",
            "Cantidad.2" : "Cant_Saldo",
            "Importe.2" : "Valor_Saldo",
            "Ubicacion Origen" : "Ubi_Origen",
            "Ubicacion Destino" : "Ubi_Destino",
            "Pedido" : "Cod_Orden_Compra",
        }
    )

    df_fin = df_fin[
        ["Fecha", "Fecha_Completa", "Cod_Albaran", "Ruc", "Proveedor", "Cod_Movimiento", "Cod_Producto", "Lote", "Precio_Unt",
         "Cant_Ingreso", "Valor_Ingreso", "Cant_Salida", "Valor_Salida", "Cant_Saldo", "Valor_Saldo", "Ubi_Origen", "Ubi_Destino",
         "Almacen", "Cod_Orden_Compra", "Cod_Bus", "Cod_Orden_Trabajo", "Tipo_Mantto"]]

    df_fin = pd.merge(
        df_fin,
        centro_costo,
        on=["Cod_Orden_Compra", "Cod_Producto"],
        how="left"
    )

    return df_fin


df_kardex_final = unificar_kardex(
    df_kardex_tatsa_2026, df_kardex_tatsa_2025, df_report_transferencias_tatsa, df_oc_centro_costo)

df_kardex_final.to_excel("Fact_Kardex_Tatsa.xlsx", index=False)


## 8. Dim_Flota

In [210]:
"""
    1. CONEXION A DF 
"""
flota = models.execute_kw(
    db, uid, password,
    "fleet.vehicle",
    "search_read",
    [
        [
           # ["default_code", "!=", False]
        ]
    ],
    {
        "fields" : ["code", "license_plate", "type_id"],
        "context" :{
            "lang": "es_PE",
            "tz": "America/Lima",
            "allowed_company_ids": [2]
        }
    }
)

df = []

for diccionario in flota:
    nuevo_diccionario = {}
        
    for clave, valor in diccionario.items():
        if isinstance(valor, list) and len(valor) == 2:
            nuevo_diccionario[clave] = valor[1]
            
        elif valor is False:
            nuevo_diccionario[clave] = None
            
        else:
            nuevo_diccionario[clave] = valor

    df.append(nuevo_diccionario)


df_flota = pd.DataFrame(df)
df_flota["code"] = df_flota["code"].fillna(df_flota["license_plate"])
df_flota["license_plate"] = df_flota["license_plate"].fillna(df_flota["code"])
df_flota = df_flota[df_flota["code"].notna()]

condicion = [
    df_flota["type_id"] =="CONVERSION",
    df_flota["type_id"] =="CHASIS/MERCEDES",
    df_flota["code"].str.contains("CHASISN", case=False, na=False),
    df_flota["type_id"] =="12 Metros",
    df_flota["type_id"] =="18 Metros",
    df_flota["type_id"] =="9 Metros",
    df_flota["code"] =="CONVERSION",
    df_flota["code"] =="CHASIS/MERCEDES",
    df_flota["code"] =="AL-080"
]

resultados = [
    "CONVERSION",
    "CHASIS/MERCEDES",
    "CHASIS/MERCEDES",
    "12 METROS",
    "18 METROS",
    "9 METROS",
    "CONVERSION",
    "CHASIS/MERCEDES",
    "AL-080"

]

df_flota["Tamaño_bus"] = np.select(condicion, resultados, default="OTROS")

df_flota = df_flota.rename(columns={
    "code": "Cod_bus",
    "license_plate" : "Placa_bus"
})

df_flota = df_flota[["Cod_bus", "Placa_bus", "Tamaño_bus"]]

df_flota_averia = df_report_averias_tatsa.copy()
df_flota_averia = df_flota_averia[["Cod_Bus_Averia", "Tamaño_Bus"]]
df_flota_averia["Placa_bus"] = df_flota_averia["Cod_Bus_Averia"]
df_flota_averia = df_flota_averia.rename(columns={
    "Cod_Bus_Averia" : "Cod_bus",
    "Tamaño_Bus" : "Tamaño_bus"
    
})

df_flota_ots = df_report_ots_tatsa.copy()
df_flota_ots = df_flota_ots[["Cod_Bus_Orden_Trabajo", "Tamaño_Bus"]]
df_flota_ots["Placa_bus"] = df_flota_ots["Cod_Bus_Orden_Trabajo"]
df_flota_ots = df_flota_ots.rename(columns={
    "Cod_Bus_Orden_Trabajo" : "Cod_bus",
    "Tamaño_Bus" : "Tamaño_bus"
    
})

bus_coa = [
    "COA-145", 
    "COA-198", 
    "COA-189",
    "COA-187-FAKE", 
    "COA-188", 
    "COA-183", 
    "COA-067", 
    "COA-043",
    "COA-101", 
    "COA-086", 
    "COA-086", 
    "COA-030", 
    "COA-105", 
    "COA-105", 
    "COA-186", 
    "COA-194", 
    "COA-176", 
    "COA-114", 
    "COA-151", 
    "COA-170"
]

tamaño_coa = [
    "REPARACIÓN DE COMPRESORA 12M LIMABUS", 
    "REPARACIÓN DE COMPRESORA 12M LIMABUS", 
    "REPARACIÓN DE COMPRESORA 12M LIMABUS", 
    "REPARACIÓN DE COMPRESORA 18M LIMABUS", 
    "REPARACIÓN DE COMPRESORA 12M LIMABUS", 
    "REPARACIÓN DE COMPRESORA 12M LIMABUS",
    "REPARACIÓN DE COMPRESORA 18M LIMABUS",
    "REPARACIÓN DE COMPRESORA 18M LIMABUS",
    "REPARACIÓN DE COMPRESORA 18M LIMABUS",
    "REPARACIÓN DE COMPRESORA 18M LIMABUS",
    "REPARACIÓN DE COMPRESORA 18M LIMABUS",
    "REPARACIÓN DE COMPRESORA 12M LIMABUS",
    "REPARACIÓN DE COMPRESORA 18M LIMABUS",
    "REPARACIÓN DE COMPRESORA 18M LIMABUS",
    "REPARACIÓN DE COMPRESORA 12M LIMABUS",
    "REPARACIÓN DE COMPRESORA 12M LIMABUS",
    "REPARACIÓN DE COMPRESORA 12M LIMABUS",
    "REPARACIÓN DE COMPRESORA 18M LIMABUS",
    "REPARACIÓN DE COMPRESORA 12M LIMABUS",
    "REPARACIÓN DE COMPRESORA 12M LIMABUS"
]

df_flota_manual = pd.DataFrame(
    {
        "Cod_bus" : bus_coa,
        "Placa_bus" : bus_coa,
        "Tamaño_bus" : tamaño_coa
    }
)


df_fin_flota = pd.concat([df_flota, df_flota_averia, df_flota_ots, df_flota_manual])
df_fin_flota = df_fin_flota.drop_duplicates(subset=["Cod_bus"], keep="first") 
df_fin_flota = df_fin_flota[df_fin_flota["Cod_bus"].notna()]
df_fin_flota.to_excel("Dim_Flota.xlsx", index=False)